# New Core: Fetching and Localizing WRF data

Replaces `Lumen get_hourly_WRF_data_from_climakitae.ipynb` and `Lumen localize gridded WRF temperatures.ipynb`. Scoped to only run on **Sacramento Executive Airport (KSAC)** for testing purposes.

**What this notebook retrieves**:

- **Localized Temperature**: fetched and bias-adjusted directly to the station via `new_core`'s `bias_adjust_model_to_station` processor (Quantile Delta Mapping against real HDP-catalog station observations). These outputs are saved in the same raw shape/units (Kelvin, native `sim` ids) as the original pre-staged `inputs/{station}.nc` files, so `Lumen detrending (new_core).ipynb` can read it with the exact same reshape/rename/unit-conversion logic as the original `Lumen detrending.ipynb`.

- **Relative Humidity**: fetched and clipped to a single gridcell without any bias adjustment. This matches the original Lumen pipeline behavior (using unadjusted gridded RH).

**What this notebook no longer retrieves:**
- Precipitation, solar irradiance, and wind speed. These were originally pulled from `get_hourly_WRF_data_from_climakitae.ipynb`, but they were never used downstream, so this notebook doesn't retrieve them.

The output files that were generated with this notebook have `(new_core)`-suffixed filenames (e.g. `inputs/Sacramento Executive Airport (KSAC) (new_core).nc`) so the original `inputs/*.nc` files and the original three notebooks remain untouched.


### Import libraries

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
from climakitae.new_core.user_interface import ClimateData
from climakitae.new_core.data_access import DataCatalog
from climakitae.new_core.processors import BiasAdjustModelToStation
from climakitae.new_core.processors import Export

cd = ClimateData(verbosity=-1)

EXPORT = True


### Station list (KSAC only)

In [ ]:
station_list = pd.read_csv("weather_stations.csv")
station_list = station_list[station_list['STATION'] == 'SACRAMENTO'].reset_index(drop=True)
resolution_to_grid_label = {45: 'd01', 9: 'd02', 3: 'd03'}

### Temperature: fetch gridded WRF data

In [ ]:
raw_temperature = {}

for station_name, station_resolution in zip(station_list['station_name'].values, station_list['resolution'].values):
    print(station_name)
    ds = (
        cd.catalog("cadcat")
        .activity_id("WRF")
        .institution_id("UCLA")
        .experiment_id(["historical", "ssp370"])
        .table_id("1hr")
        .grid_label(resolution_to_grid_label[station_resolution])
        .variable_id("t2")
        .processes({
            # keep full ensemble, since `simulation_list.csv` includes flags 
            # that specify which simulations to use in the detrending notebook
            "filter_unadjusted_models": "no",
        })
        .get()
    )
    raw_temperature[station_name] = ds

del ds


### Temperature: Bias-adjust the gridded data to the station

Separating localization step from data retrieval step.

In [ ]:
catalog = DataCatalog()

for station_name in station_list['station_name'].values:
    print(station_name)
    station_code = station_name.split('(')[-1].rstrip(')')

    # Creating the BiasAdjustModelToStation processor and executing it on the raw temperature data for the station.
    # This will apply Quantile Delta Mapping to adjust the model data to match the station's observed data.
    bias_adjust = BiasAdjustModelToStation({"stations": [station_code]})
    bias_adjust.set_data_accessor(catalog)
    ds = bias_adjust.execute(raw_temperature[station_name], context={})

    # bias_adjust_model_to_station returns one data variable per station, named by HDP's
    # display name, so rename it to match weather_stations.csv's station_name so the
    # detrending notebook can open this file exactly like the original inputs/{station}.nc
    station_var = list(ds.data_vars)[0]
    final_ds = ds.rename({station_var: station_name})

    if EXPORT:
        ds.to_netcdf(f'inputs/{station_name} (new_core).nc')
        del raw_temperature, ds


### Temperature: Exporting the data to be used in `detrending.ipynb`

In [ ]:
### Export your data, depending on whether or not you localized it
# temp_filename = "inputs/{station_name}_temp_(new_core).nc"
localized_filename = f"inputs/{station_name}_temp_localized_(new_core).nc"
export_proc = Export({
    "filename": localized_filename
})
export_proc.execute(ds, context={})

### Relative humidity: nearest-gridcell clip (no bias adjustment)

Grabbing relative humidity at the nearest gridcell to the station itself. This is consistent behavior with the Lumen notebooks for grabbing unadjusted gridded data.

In [ ]:
station_row = station_list.iloc[0]

rh_ds = (
    cd
    .reset()
    .catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .experiment_id(["historical", "ssp370"])
    .table_id("1hr")
    .grid_label(resolution_to_grid_label[station_row['resolution']])
    .variable_id("relative_humidity_2m")
    .processes({
        "clip": (station_row['LAT_Y'], station_row['LON_X']),
        # keep full ensemble, since `simulation_list.csv` includes flags 
        # that specify which simulations to use in the detrending notebook
        "filter_unadjusted_models": "no",
        "export": {
            "filename": f"inputs/{station_row['station_name']}_rh_(new_core).nc"
        }
    })
    .get()
)
